In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/DX PROJ./유튜브 크롤링/층간소음_youtube_clean.csv")
df

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 코랩 konlpy 실행
!curl -s https://raw.githubusercontent.com/teddylee777/machine-learning/master/99-Misc/01-Colab/mecab-colab.sh | bash

# Mecab 설치 후 Google Drive에 복사
!cp -r /usr/local/lib/mecab /content/drive/MyDrive/mecab
!cp -r /usr/local/etc/mecabrc /content/drive/MyDrive/mecab

In [ ]:
! pip install kiwipiepy
import pandas as pd
import re
from konlpy.tag import Okt
from kiwipiepy import Kiwi

# 20자 이하 삭제

In [ ]:
import pandas as pd

# contents_clean 열의 길이가 20자 초과인 행만 선택하여 다시 저장
df = df[df['contents_clean'].str.len() > 22]

In [ ]:
df

# 중복글 제거

In [ ]:
df["contents_clean"].duplicated()
df.drop_duplicates("contents_clean")

# 대소문자 통일

In [ ]:
df["contents_clean"] = df["contents_clean"].apply(lambda x : x.lower())
df

# 토크나이징

In [ ]:
okt = Okt()
df["token"] = df["contents_clean"].apply(lambda x : okt.morphs(x, stem = True, norm = True))
df

In [ ]:
from tqdm import tqdm

okt = Okt()
tqdm.pandas()

def extract_pos(text, pos):
  left=[]
  for i in okt.pos(text,stem=True, norm =True):  # i 가 하나의 튜플 형성
    if i[1] in pos:   # 만약 i의 첫번째 원소가 명사 동사 형용사라면,
      left.append(i[0])
  return left

df["token"] = df["contents_clean"].progress_apply(lambda x : extract_pos(x, ["Noun", "Verb", "Adjective"]))
df

# 불용어 제거

In [ ]:
from kiwipiepy.utils import Stopwords

stopwords = Stopwords()
sw = set([i[0] for i in stopwords.stopwords]) # 중복 제거 위해 set 사용

cleaned_token = []
for i in df["token"]:
  imsi = []
  for w in i:
    if w not in sw:
      imsi.append(w)
  cleaned_token.append(imsi)
df["token"] = cleaned_token
df

In [ ]:
# 특정 컬럼 제거
# errors='ignore'를 추가하면 해당 컬럼이 데이터프레임에 없더라도 오류 없이 실행됩니다.
df = df.drop(['author', 'title','contents','date'], axis=1, errors='ignore')

# 컬럼이 잘 제거되었는지 확인
print(df.columns)

In [ ]:
df

In [ ]:
df.to_csv("유튜브 전처리 후 토크나이징까지.csv", index=False, encoding='utf-8-sig')

print(" 데이터 저장 완료!")